# Aesha's contribution

In [1]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install transformers datasets peft accelerate evaluate bert_score rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 27.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=5a20f657e6c5268eca49f68a0d1cb465305951dc42a11211d8f37913d4b5a396
  Stored in directory: /Users/aesha/Library/Caches/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
Note: you may need to restart the kernel to use updated packages.


In [3]:
import optuna
import pandas as pd
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import Dataset

# Load training passages and test QA data
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Sample data
train_data_sample = passages_df.sample(frac=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_data_sample[['passage']].rename(columns={'passage': 'text'}))
test_dataset = Dataset.from_pandas(test_df[['question']].rename(columns={'question': 'text'}))

# Load GPT-2 model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set the pad token to eos_token if not set already
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos_token

# Apply LoRA (Low-Rank Adaptation) to the model
lora_config = LoraConfig(
    r=8,  # Rank of the low-rank adaptation
    lora_alpha=16,  # Scaling factor
    target_modules=["attn.c_attn", "attn.c_proj"],  # LoRA applied to these layers for GPT-2
    lora_dropout=0.1,  # Dropout for LoRA
)

# Apply LoRA to the GPT-2 model
model_lora = get_peft_model(model, lora_config)

# Tokenization function
def tokenize_data(examples):
    tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True, max_length=256, return_attention_mask=True)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Tokenize the passages and test data
train_data = train_dataset.map(tokenize_data, batched=True)
test_data = test_dataset.map(tokenize_data, batched=True)

# Define the function for the Optuna objective
def objective(trial):
    # Hyperparameter search space
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)
    batch_size = trial.suggest_int('batch_size', 1, 8)  # Use smaller batch size
    num_train_epochs = trial.suggest_int('num_train_epochs', 3, 10)

    # Training Arguments with gradient accumulation and mixed precision enabled
    training_args = TrainingArguments(
        output_dir='./results',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=500,
        gradient_accumulation_steps=4,  # Gradient accumulation for larger effective batch size
        fp16=True,  # Mixed precision training
    )

    # Trainer
    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=test_data,
    )

    # Train the model
    trainer.train()

    # Evaluate the model
    results = trainer.evaluate()
    return results['eval_loss']  # Minimize the evaluation loss

# Create the Optuna study and optimize the hyperparameters
study = optuna.create_study(direction='minimize')  # Minimize the evaluation loss
study.optimize(objective, n_trials=5)  # You can change `n_trials` to test more hyperparameter sets

# Get the best trial (the optimal hyperparameters)
best_trial = study.best_trial
print("Best trial:")
print("  Value: ", best_trial.value)
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

2025-07-23 22:49:21.008652: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

/opt/anaconda3/lib/python3.12/site-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/4022 [00:00<?, ? examples/s]

Map:   0%|          | 0/4719 [00:00<?, ? examples/s]

[I 2025-07-23 22:50:32,336] A new study created in memory with name: no-name-f5278a9c-ca7c-4e7e-ae83-2ac21f9591ab
/var/folders/wb/n14v6l71017dfdm43tgbdl780000gn/T/ipykernel_19127/1461634556.py:51: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)
[W 2025-07-23 22:50:34,886] Trial 0 failed with parameters: {'learning_rate': 2.3297723678597798e-05, 'batch_size': 6, 'num_train_epochs': 4} because of the following error: ValueError('Tried to use `fp16` but it is not supported on cpu').
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/wb/n14v6l71017dfdm43tgbdl780000gn/T/ipykernel_19127/146163

ValueError: Tried to use `fp16` but it is not supported on cpu

In [ ]:
# Best hyperparameters
best_lr = best_trial.params['learning_rate']
best_batch_size = best_trial.params['batch_size']
best_epochs = best_trial.params['num_train_epochs']

# Final model training with the best hyperparameters
final_training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=best_lr,
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    num_train_epochs=best_epochs,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
)

final_trainer = Trainer(
    model=model_lora,
    args=final_training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

final_trainer.train()

# Evaluate the model on the test set
final_results = final_trainer.evaluate()
print("Final Evaluation Results:", final_results)

# LLM Used
1. Perform Model tuning on RAG approach on pretained LLM.
2. Provide step by step solution from scratch starting from importing data to the final evaluation results.
3. Facing error the following error. what's the solution to overcome this:Tried to use `fp16` but it is not supported on cpu